In [ ]:
!pip install fastapi pydantic sqlalchemy aiosqlite datasets trl peft accelerate pyyaml numpy pytest-asyncio


In [ ]:
!git clone https://github.com/YOUR_TEAM/triage-backend
import sys
sys.path.insert(0, '/content/triage-backend')


In [ ]:
from triage.training.unsloth_trainer import setup_model
model, tokenizer = setup_model('unsloth/gemma-3-4b-it-unsloth-bnb-4bit')
print(model, tokenizer)


In [ ]:
from triage.training.trajectory_collector import TrajectoryCollector
collector = TrajectoryCollector(output_dir='/content/triage_data')
baseline_trajectories = await collector.collect(n_episodes=3)
baseline_rewards = [trajectory.total_reward for trajectory in baseline_trajectories]
baseline_rewards


In [ ]:
from triage.training.preference_labeler import PreferenceLabeler
labeler = PreferenceLabeler()
pairs = labeler.label_trajectories(baseline_trajectories)
dataset = labeler.export_as_hf_dataset(pairs, '/content/triage_data/preference_dataset.json')
len(pairs)


In [ ]:
from triage.training.dpo_trainer import DPOConfig, TRIAGEDPOTrainer
trainer = TRIAGEDPOTrainer(DPOConfig(data_dir='/content/triage_data', output_dir='/content/triage_model', mock_mode=True))
metrics = await trainer.train()
metrics
